# Package.json

In [187]:
from os.path import join
from bs4 import BeautifulSoup, NavigableString
from copy import deepcopy
from re import split, sub
import pandas as pd

REPO_ROOT_ABSPATH = "/utkusarioglu-com/workshops/backtesting-workshop"
assets_abspath = join(REPO_ROOT_ABSPATH, "assets/npm")
artifacts_abspath = join(REPO_ROOT_ABSPATH, "artifacts/npm")
html_abspath = join(assets_abspath, "package-json.html")
csv_target = join(artifacts_abspath, "package_json.csv")

In [53]:
html = open(html_abspath, "r").read()

<main class="Box-sc-g0xbh4-0 jrNUvm"><div class="Box-sc-g0xbh4-0 goytIH"><nav aria-label="Breadcrumbs" class="Breadcrumbs__BreadcrumbsBase-sc-9m4wsf-1 jGmqUI"><ol class="Box-sc-g0xbh4-0 eItYAW"><li class="Breadcrumbs__Wrapper-sc-9m4wsf-0 bpSOTI"><a class="Breadcrumbs__BreadcrumbsItem-sc-9m4wsf-2 dKiiRk" href="/cli">CLI</a></li><li class="Breadcrumbs__Wrapper-sc-9m4wsf-0 bpSOTI"><a class="Breadcrumbs__BreadcrumbsItem-sc-9m4wsf-2 dKiiRk" href="/cli/v10/configuring-npm">Configuring</a></li><li class="Breadcrumbs__Wrapper-sc-9m4wsf-0 bpSOTI"><a aria-current="page" class="Breadcrumbs__BreadcrumbsItem-sc-9m4wsf-2 dKiiRk selected" href="/cli/v10/configuring-npm/package-json" selected="">package.json</a></li></ol></nav><h1 class="components__StyledHeading-sc-13rww2g-0 components__h1-sc-13rww2g-1 cZjJlu dMhHzi components__StyledHeading-sc-13rww2g-0 cZjJlu">package.json</h1><div class="Box-sc-g0xbh4-0 iLGAbu">Specifics of npm's package.json handling</div></div><div class="Box-sc-g0xbh4-0 skip-na

In [195]:
sections = {}
current = {
    "header_string": None,
    "parts": BeautifulSoup("", "html.parser"),
    "header": None,
    "parts_length": 0,
}
child_count = 0
page_url = "https://docs.npmjs.com/cli/v10/configuring-npm/package-json"

soup = BeautifulSoup(html, "html.parser")
for child in soup.main.children:
    child_count += 1
    if child.name == None:
        continue

    del child["class"]
    del child["style"]
    if child.svg is not None:
        child.svg.decompose()
    for button in child.find_all("button"):
        button.decompose()
    for sub in child.descendants:
        if sub.name is not None:
            del sub["class"]
            del sub["style"]

    for anchor in child.find_all("a"):
        if anchor["href"].startswith("#"):
            anchor["href"] = page_url + anchor["href"]

    if child.name == "h3":
        if current["header"] is not None:
            current["parts_pretty"] = current["parts"].prettify()
            summary = BeautifulSoup(
                "\n".join([str(e) for e in current["parts"].find_all("p")]),
                "html.parser",
            )
            summary = str(summary.text)
            summary = summary.replace(current["header_string"], "___")
            summary = split(r"[\.:]\s", summary)[0] + "."
            current["summary"] = f"<p>{summary}</p>"
            sections[current["header_string"]] = current
        current = {
            "header": child.prettify(),
            "header_string": child.get_text(),
            "parts": BeautifulSoup("", "html.parser"),
            "parts_length": 0,
        }
        continue
    current["parts"].append(child)
    current["parts_length"] += 1

del sections["Description"]
for section in list(sections.values())[2::5]:
    print(section["header_string"], section["summary"], sep="\n")
    print("\n")

description
<p>Put a ___ in it.</p>


people fields: author, contributors
<p>The "author" is one person.</p>


bin
<p>A lot of packages have one or more executable files that they'd like to install into the PATH.</p>


config
<p>A "___" object can be used to set ___uration parameters used in package scripts that persist across upgrades.</p>


bundleDependencies
<p>This defines an array of package names that will be bundled when publishing the package.</p>


cpu
<p>If your code only runs on certain ___ architectures, you can specify which ones.</p>




In [196]:
df = pd.DataFrame(sections.values())
df.rename(
    columns={
        "header": "HeaderElem",
        "header_string": "HeaderString",
        # "parts": "PartsElem",
        "summary": "Summary",
        "parts_pretty": "PartsElem",
    },
    inplace=True,
)
df.drop(["parts_length", "parts"], axis=1, inplace=True)
df["Tags"] = "Version-10.8.2 WebScraped"
df

,HeaderElem,HeaderString,PartsElem,Summary,Tags
0,"<h3 id=""name"">\n <a aria-label=""name permalink...",name,"<p>\n If you plan to publish your package, the...","<p>If you plan to publish your package, the mo...",Version-10.8.2 WebScraped
1,"<h3 id=""version"">\n <a aria-label=""version per...",version,"<p>\n If you plan to publish your package, the...","<p>If you plan to publish your package, the mo...",Version-10.8.2 WebScraped
2,"<h3 id=""description-1"">\n <a aria-label=""descr...",description,<p>\n Put a description in it. It's a string. ...,<p>Put a ___ in it.</p>,Version-10.8.2 WebScraped
3,"<h3 id=""keywords"">\n <a aria-label=""keywords p...",keywords,<p>\n Put keywords in it. It's an array of str...,<p>Put ___ in it.</p>,Version-10.8.2 WebScraped
4,"<h3 id=""homepage"">\n <a aria-label=""homepage p...",homepage,<p>\n The URL to the project homepage.\n</p>\n...,<p>The URL to the project ___.</p>,Version-10.8.2 WebScraped
5,"<h3 id=""bugs"">\n <a aria-label=""bugs permalink...",bugs,<p>\n The URL to your project's issue tracker ...,<p>The URL to your project's issue tracker and...,Version-10.8.2 WebScraped
6,"<h3 id=""license"">\n <a aria-label=""license per...",license,<p>\n You should specify a license for your pa...,<p>You should specify a ___ for your package s...,Version-10.8.2 WebScraped
7,"<h3 id=""people-fields-author-contributors"">\n ...","people fields: author, contributors","<p>\n The ""author"" is one person. ""contributor...","<p>The ""author"" is one person.</p>",Version-10.8.2 WebScraped
8,"<h3 id=""funding"">\n <a aria-label=""funding per...",funding,<p>\n You can specify an object containing a U...,<p>You can specify an object containing a URL ...,Version-10.8.2 WebScraped
9,"<h3 id=""files"">\n <a aria-label=""files permali...",files,<p>\n The optional\n <code>\n files\n </code>...,<p>The optional ___ field is an array of file ...,Version-10.8.2 WebScraped


In [197]:
df.to_csv(csv_target, sep="|", index=False, header=False)